In [1]:
import requests
from datetime import datetime, timedelta
from fastapi import FastAPI, HTTPException
import joblib
import numpy as np

In [2]:
# 固定开始时间
start_time = datetime(2025, 5, 1)  # 2025年5月1日 00:00:00

# 结束时间为当前时间
end_time = datetime.now()

In [3]:
names = [
        'DLDZ_DQ200_SYSTEM_PI05.PV',
        'DLDZ_AVS_SYSTEM_PI05.PV',
        'DLDZ_DQ200_LLJ01_FI01.PV',
        'DLDZ_AVS_LLJ01_FI01.PV'
        ]

In [5]:
# 构建API请求参数
params = {
    "startTime": start_time.isoformat(timespec='milliseconds'),
    "endTime": end_time.isoformat(timespec='milliseconds'),
    "interval": 60000,  # 1小时=3600000ms, 1分钟=60000ms
    "valueonly": 0,
    "decimal": 2,
    "names": ','.join(names)
}

In [6]:
# 发送GET请求
try:
    response = requests.get(
        "http://8.130.25.118:8000/api/hisdata",
        params=params,
        timeout=10  # 设置超时时间
    )
    response.raise_for_status()  # 检查HTTP状态码
except Exception as e:
    raise Exception(f"API请求失败: {str(e)}")
# 解析响应数据
data = response.json()

In [7]:
# 解析数据
items = data.get('items', [])

In [8]:
import pandas as pd

# 构建 DataFrame
timestamp_dict = {}
columns = []

for item in items:
    var_name = item['name']
    columns.append(var_name)
    
    for val in item['vals']:
        timestamp = val['time']
        value = val['val']
        
        if timestamp not in timestamp_dict:
            timestamp_dict[timestamp] = {}
        
        timestamp_dict[timestamp][var_name] = value

# 转换为 DataFrame
df = pd.DataFrame([
    {'timestamp': ts, **values} for ts, values in timestamp_dict.items()
])

# 时间戳转为可读格式（如果需要）
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [9]:
df.head()

,timestamp,DLDZ_DQ200_SYSTEM_PI05.PV,DLDZ_AVS_SYSTEM_PI05.PV,DLDZ_DQ200_LLJ01_FI01.PV,DLDZ_AVS_LLJ01_FI01.PV
0,2025-05-01 00:00:00,6.07,6.40,49.25,78.77
1,2025-05-01 00:01:00,6.14,6.37,49.08,81.23
2,2025-05-01 00:02:00,6.23,6.53,46.98,85.01
3,2025-05-01 00:03:00,6.30,6.79,48.79,83.35
4,2025-05-01 00:04:00,6.28,6.70,52.61,83.35


# 数据清洗

In [10]:
# 处理异常值
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return (series < lower_bound) | (series > upper_bound)

column = ['DLDZ_DQ200_SYSTEM_PI05.PV', 'DLDZ_AVS_SYSTEM_PI05.PV',
       'DLDZ_DQ200_LLJ01_FI01.PV', 'DLDZ_AVS_LLJ01_FI01.PV']

# 对DataFrame中的每一个数值列进行处理
for column in df.select_dtypes(include=[np.number]).columns:
    outliers_mask = detect_outliers_iqr(df[column])
    # 将异常值替换为NaN
    df.loc[outliers_mask, column] = np.nan
    # 使用线性插值填补NaN（包括原始缺失值和被标记的异常值）
    df[column] = df[column].interpolate(method='linear')

In [13]:
# 处理缺失值
df_cleaned = df.dropna()

In [17]:
df_cleaned.head()

,timestamp,DLDZ_DQ200_SYSTEM_PI05.PV,DLDZ_AVS_SYSTEM_PI05.PV,DLDZ_DQ200_LLJ01_FI01.PV,DLDZ_AVS_LLJ01_FI01.PV
371,2025-05-01 06:11:00,6.42,6.54,73.1500,4.77
372,2025-05-01 06:12:00,6.44,6.54,73.4075,4.28
373,2025-05-01 06:13:00,6.43,6.54,73.6650,5.20
374,2025-05-01 06:14:00,6.41,6.54,73.9225,5.24
375,2025-05-01 06:15:00,6.37,6.54,74.1800,4.40


In [19]:
# 特征工程
df_cleaned['瞬时流量'] = df_cleaned['DLDZ_DQ200_LLJ01_FI01.PV'] + df_cleaned['DLDZ_AVS_LLJ01_FI01.PV']
df_cleaned['总压力'] = df_cleaned['DLDZ_DQ200_SYSTEM_PI05.PV'] + df_cleaned['DLDZ_AVS_SYSTEM_PI05.PV']

C:\Users\Svill\AppData\Local\Temp\ipykernel_19100\967827121.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['瞬时流量'] = df_cleaned['DLDZ_DQ200_LLJ01_FI01.PV'] + df_cleaned['DLDZ_AVS_LLJ01_FI01.PV']
C:\Users\Svill\AppData\Local\Temp\ipykernel_19100\967827121.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['总压力'] = df_cleaned['DLDZ_DQ200_SYSTEM_PI05.PV'] + df_cleaned['DLDZ_AVS_SYSTEM_PI05.PV']


In [20]:
df_cleaned.head()

,timestamp,DLDZ_DQ200_SYSTEM_PI05.PV,DLDZ_AVS_SYSTEM_PI05.PV,DLDZ_DQ200_LLJ01_FI01.PV,DLDZ_AVS_LLJ01_FI01.PV,瞬时流量,总压力
371,2025-05-01 06:11:00,6.42,6.54,73.1500,4.77,77.9200,12.96
372,2025-05-01 06:12:00,6.44,6.54,73.4075,4.28,77.6875,12.98
373,2025-05-01 06:13:00,6.43,6.54,73.6650,5.20,78.8650,12.97
374,2025-05-01 06:14:00,6.41,6.54,73.9225,5.24,79.1625,12.95
375,2025-05-01 06:15:00,6.37,6.54,74.1800,4.40,78.5800,12.91


In [ ]:
# 保存为 Excel 文件
output_file = 'new_history_data.xlsx'
df_cleaned.to_excel(output_file, index=False)

print(f"数据已成功保存到: {output_file}")

数据已成功保存到: his_data_output.xlsx
